# Graph Theory for Visual Learners

In [ ]:
import time
from dataclasses import dataclass, field


@dataclass
class Canvas:
    background_color: str = "#ffffff"


@dataclass
class Camera:
    center: tuple[float, float, float] = (0.0, 0.0, 0.0)
    height: float = 480.0
    _start_center: tuple[float, float, float] = field(default=(0.0, 0.0, 0.0), init=False, repr=False)
    _target_center: tuple[float, float, float] = field(default=(0.0, 0.0, 0.0), init=False, repr=False)
    _start_height: float = field(default=480.0, init=False, repr=False)
    _target_height: float = field(default=480.0, init=False, repr=False)
    _start_time: float = field(default=0.0, init=False, repr=False)
    _duration: float = field(default=0.0, init=False, repr=False)
    _moving: bool = field(default=False, init=False, repr=False)

    @staticmethod
    def _zoom_from_z(z: float) -> float:
        # In this 2D visualizer, z acts as camera distance, which maps to visible world height.
        return max(60.0, abs(float(z)))

    def _sample(self, now: float | None = None):
        if not self._moving:
            return self.center

        if now is None:
            now = time.perf_counter()

        progress = (now - self._start_time) / self._duration
        if progress >= 1.0:
            self.center = self._target_center
            self.height = self._target_height
            self._moving = False
            return self.center

        progress = max(0.0, min(1.0, progress))
        eased = progress * progress * (3.0 - 2.0 * progress)

        sx, sy, sz = self._start_center
        tx, ty, tz = self._target_center
        self.center = (
            sx + (tx - sx) * eased,
            sy + (ty - sy) * eased,
            sz + (tz - sz) * eased,
        )
        self.height = self._start_height + (self._target_height - self._start_height) * eased
        return self.center

    def move_to(self, x: float, y: float, z: float, duration: float = 1.2):
        current = self._sample()
        self.center = current
        self._start_center = current
        self._target_center = (float(x), float(y), float(z))
        self._start_height = float(self.height)
        self._target_height = self._zoom_from_z(z)
        self._start_time = time.perf_counter()
        self._duration = max(0.01, float(duration))
        self._moving = True
        return self

    def state(self):
        cx, cy, cz = self._sample()
        return {
            "center": [cx, cy, cz],
            "height": float(self.height),
            "moving": bool(self._moving),
        }


canvas = Canvas()

camera = Camera()

In [ ]:
import math
import time
from dataclasses import dataclass, field


@dataclass
class NodeMeta:
    fill_color: str = "#e8c547"
    stroke_color: str = "#1f2937"
    stroke_width: float = 3.0
    diameter: float = 50.0
    label_size: float = 30.0
    label_color: str = "#111827"
    display_label: bool = True


@dataclass
class Node:
    label: str
    metadata: NodeMeta = field(default_factory=NodeMeta)
    pos: tuple[float, float] | None = None
    _start_pos: tuple[float, float] = field(default=(0.0, 0.0), init=False, repr=False)
    _target_pos: tuple[float, float] = field(default=(0.0, 0.0), init=False, repr=False)
    _start_time: float = field(default=0.0, init=False, repr=False)
    _duration: float = field(default=0.0, init=False, repr=False)
    _moving: bool = field(default=False, init=False, repr=False)

    @staticmethod
    def _parse_pos(raw_pos):
        if isinstance(raw_pos, (list, tuple)) and len(raw_pos) >= 2:
            try:
                x = float(raw_pos[0])
                y = float(raw_pos[1])
            except Exception:
                return None
            if math.isfinite(x) and math.isfinite(y):
                return (x, y)
            return None

        if isinstance(raw_pos, dict):
            try:
                x = float(raw_pos.get("x"))
                y = float(raw_pos.get("y"))
            except Exception:
                return None
            if math.isfinite(x) and math.isfinite(y):
                return (x, y)
            return None

        return None

    def _sample(self, now: float | None = None):
        if not self._moving:
            return self._parse_pos(self.pos)

        if now is None:
            now = time.perf_counter()

        progress = (now - self._start_time) / self._duration
        if progress >= 1.0:
            self.pos = self._target_pos
            self._moving = False
            return self._target_pos

        progress = max(0.0, min(1.0, progress))
        eased = progress * progress * (3.0 - 2.0 * progress)
        sx, sy = self._start_pos
        tx, ty = self._target_pos
        current = (
            sx + (tx - sx) * eased,
            sy + (ty - sy) * eased,
        )
        self.pos = current
        return current

    def move_to(self, x: float, y: float, duration: float = 1.2):
        target = (float(x), float(y))
        if not (math.isfinite(target[0]) and math.isfinite(target[1])):
            raise ValueError("x and y must be finite numbers")

        duration_value = float(duration)
        if not math.isfinite(duration_value):
            raise ValueError("duration must be a finite number")

        current = self._sample()
        if current is None:
            current = self._parse_pos(self.pos)
        if current is None:
            current = self._target_pos
            self.pos = current

        self._start_pos = current
        self._target_pos = target
        self._start_time = time.perf_counter()
        self._duration = max(0.01, duration_value)
        self._moving = self._start_pos != self._target_pos
        if not self._moving:
            self.pos = self._target_pos
        return self

    def state(self):
        pos = self._sample()
        return {
            "pos": [pos[0], pos[1]] if pos is not None else None,
            "moving": bool(self._moving),
        }


@dataclass
class EdgeMeta:
    color: str = "#8a8a8a"
    width: float = 4.0
    display_label: bool = False


@dataclass
class Edge:
    frm: Node
    to: Node
    label: str = ""
    metadata: EdgeMeta = field(default_factory=EdgeMeta)


@dataclass
class GraphMeta:
    label: str = ""
    layout: str = "force"
    display_label: bool = False


class Graph:
    def __init__(self, nodes, edges, metadata=None):
        self.metadata = metadata if isinstance(metadata, GraphMeta) else GraphMeta()
        self.nodes = [
            node if isinstance(node, Node) else Node(label=node)
            for node in nodes
        ]
        self.edges = [
            edge if isinstance(edge, Edge) else Edge(
                frm=edge[0] if isinstance(edge[0], Node) else Node(label=edge[0]),
                to=edge[1] if isinstance(edge[1], Node) else Node(label=edge[1]),
            )
            for edge in edges
        ]

    def __str__(self):
        node_labels = ", ".join(node.label for node in self.nodes) or "(none)"
        edge_lines = [f"  - {edge.frm.label} -> {edge.to.label}" for edge in self.edges]
        edges_block = "\n".join(edge_lines) if edge_lines else "  - (none)"
        return f"Graph:\n  Nodes: {node_labels}\n  Edges:\n{edges_block}"

    def __repr__(self):
        return self.__str__()

In [ ]:
graph_1 = Graph(
    nodes=["a", "b", "c", "d", "e"],
    edges=[("a", "b"), ("b", "c"), ("c", "d"), ("d", "e"), ("e", "a"),
           ("a", "c"), ("a", "d")]
)

In [ ]:
graph_1.nodes

In [ ]:
graph_2 = Graph(
    nodes=["a", "b"],
    edges=[("a", "b"), ("b", "c")]
)